# PAN-AI Google Colab Experiment Notebook (Local Qwen + Ollama)

Notebook ini digunakan untuk menguji model AI PANDAI (FastAPI + OpenClaw + Hermes Memory) menggunakan GPU/CPU Google Colab langsung melalui VS Code. 

Sistem ini menggunakan model **Qwen lokal (via Ollama)** yang berjalan 100% gratis di lingkungan cloud Colab, tanpa menggunakan Google Gemini API.

## 1. Instalasi Ollama & Dependensi di Colab

Jalankan sel ini untuk menginstal Ollama dan pustaka-pustaka Python yang diperlukan di dalam Google Colab VM.

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Install Python dependencies
!pip install fastapi uvicorn pydantic requests pyyaml pyngrok paho-mqtt

## 2. Menjalankan Layanan Ollama & Pull Model Qwen

Sel ini akan menjalankan daemon Ollama di latar belakang (*background process*) dan mengunduh model **Qwen 2.5** (atau model Qwen pilihan Anda).

In [ ]:
import subprocess
import time

print("[PAN-AI] Menjalankan daemon Ollama di background...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(3) # Tunggu daemon boot

print("[PAN-AI] Mengunduh model Qwen (qwen2.5:7b)... Ini membutuhkan waktu beberapa menit.")
# Anda juga bisa menggantinya dengan model qwen lainnya sesuai spesifikasi GPU Colab
!ollama pull qwen2.5:7b

## 3. Pembuatan Struktur Folder & Kode Server

Sel ini menulis berkas-berkas Python kustom (seperti `digital_twin`, `evaluate_biometrics`, dan `generate_quiz` berbasis Ollama) di lingkungan Colab.

In [ ]:
import os
os.makedirs("skills", exist_ok=True)
os.makedirs("memories", exist_ok=True)

# Menulis skills/digital_twin.py
with open("skills/digital_twin.py", "w", encoding="utf-8") as f:
    f.write('''import os
from typing import Dict, Optional
MEMORIES_DIR = "memories"
DEFAULT_USER_TEMPLATE = "# Student Profile\\n- Learning Style: Visual\\n- Baseline Heart Rate: 75 bpm\\n"
DEFAULT_MEMORY_TEMPLATE = "# Learning Log\\n- Current Chapter: Chapter 1\\n- Fatigue Count: 0\\n"

def load_student_memory(student_id: str) -> Dict[str, str]:
    student_dir = os.path.join(MEMORIES_DIR, student_id)
    os.makedirs(student_dir, exist_ok=True)
    u_path = os.path.join(student_dir, "USER.md")
    m_path = os.path.join(student_dir, "MEMORY.md")
    if not os.path.exists(u_path):
        with open(u_path, "w") as f: f.write(DEFAULT_USER_TEMPLATE)
    if not os.path.exists(m_path):
        with open(m_path, "w") as f: f.write(DEFAULT_MEMORY_TEMPLATE)
    with open(u_path, "r") as f: u_c = f.read()
    with open(m_path, "r") as f: m_c = f.read()
    return {"user_profile": u_c, "learning_log": m_c}

def update_student_memory(student_id: str, user_profile: Optional[str] = None, learning_log: Optional[str] = None) -> bool:
    student_dir = os.path.join(MEMORIES_DIR, student_id)
    os.makedirs(student_dir, exist_ok=True)
    if user_profile:
        with open(os.path.join(student_dir, "USER.md"), "w") as f: f.write(user_profile)
    if learning_log:
        with open(os.path.join(student_dir, "MEMORY.md"), "w") as f: f.write(learning_log)
    return True
''')

# Menulis skills/evaluate_biometrics.py
with open("skills/evaluate_biometrics.py", "w", encoding="utf-8") as f:
    f.write('''def evaluate_student_state(heart_rate: float, hrv: float, ear: float, gsr: float, focus_score: float):
    state = "NORMAL"
    reason = "Kondisi stabil."
    if ear < 0.20 or focus_score < 40:
        state = "MENGANTUK"
        reason = "Deteksi kedipan mata lambat."
    elif heart_rate > 95.0 or hrv < 40.0:
        state = "STRES"
        reason = "Detak jantung tinggi."
    return {
        "state": state,
        "reason": reason,
        "biometrics_snapshot": {"heart_rate": heart_rate, "hrv": hrv, "ear": ear, "gsr": gsr, "focus_score": focus_score}
    }
''')

# Menulis skills/generate_quiz.py (Menggunakan Ollama lokal)
with open("skills/generate_quiz.py", "w", encoding="utf-8") as f:
    f.write('''import requests
import json
def generate_educational_quiz(topic: str, grade: str, difficulty: str, num_questions: int = 5, model_name: str = "qwen2.5:7b"):
    url = "http://localhost:11434/api/chat"
    prompt = (
        f"Buatlah kuis pilihan ganda tentang '{topic}' untuk kelas '{grade}' dengan tingkat kesulitan '{difficulty}'. "
        f"Buat tepat {num_questions} soal. Setiap soal harus memiliki tepat 4 opsi pilihan (A, B, C, D). "
        "Kembalikan jawaban dalam format JSON terstruktur yang berisi array 'questions'. "
        "Setiap item soal wajib memiliki properti: 'question' (string), 'options' (array berisi 4 string), "
        "'answer' (string berisi A, B, C, atau D), dan 'explanation' (string penjelasan detail). "
        "Tulis semua konten dalam Bahasa Indonesia."
    )
    payload = {
        "model": model_name,
        "messages": [
            {
                "role": "system",
                "content": "Anda adalah asisten pembuat soal ujian sekolah yang ahli. Anda wajib selalu mengembalikan output dalam format JSON yang valid."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        "format": "json",
        "stream": False
    }
    try:
        r = requests.post(url, json=payload, headers={"Content-Type": "application/json"}, timeout=60)
        r.raise_for_status()
        text = r.json()["message"]["content"]
        return json.loads(text)
    except Exception as e:
        return {"error": str(e)}
''')

# Menulis server_app.py
with open("server_app.py", "w", encoding="utf-8") as f:
    f.write('''import os
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from skills.digital_twin import load_student_memory, update_student_memory
from skills.evaluate_biometrics import evaluate_student_state
from skills.generate_quiz import generate_educational_quiz

app = FastAPI()

class TelemetryPayload(BaseModel):
    studentId: str
    heart_rate: float
    hrv: float
    ear: float
    gsr: float
    focus_score: float

class QuizPayload(BaseModel):
    topic: str
    grade: str
    difficulty: str
    num_questions: int = 5
    model_name: str = "qwen2.5:7b"

@app.post("/api/evaluate-biometrics")
def evaluate_biometrics(payload: TelemetryPayload):
    try:
        mem = load_student_memory(payload.studentId)
        eval_res = evaluate_student_state(
            payload.heart_rate, payload.hrv, payload.ear, payload.gsr, payload.focus_score
        )
        state = eval_res["state"]
        if state != "NORMAL":
            new_log = mem["learning_log"].strip() + f"\\n- [{state}] {eval_res['reason']}"
            update_student_memory(payload.studentId, learning_log=new_log)
        return {
            "success": True,
            "student_id": payload.studentId,
            "cognitive_state": state,
            "reason": eval_res["reason"],
            "intervention_recommended": state != "NORMAL"
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/api/generate-quiz")
def generate_quiz(payload: QuizPayload):
    res = generate_educational_quiz(payload.topic, payload.grade, payload.difficulty, payload.num_questions, payload.model_name)
    if "error" in res:
        raise HTTPException(status_code=500, detail=res["error"])
    return res
''')
print("[PAN-AI] Folder dan berkas server berhasil disiapkan.")

## 4. Konfigurasi Terowongan (Ngrok Tunneling)

Masukkan **Ngrok Authtoken** Anda di bawah untuk mengaktifkan terowongan HTTPS ke server FastAPI Colab Anda.

In [ ]:
from pyngrok import ngrok

NGROK_AUTHTOKEN = "YOUR_NGROK_AUTHTOKEN_HERE"
ngrok.set_auth_token(NGROK_AUTHTOKEN)

public_url = ngrok.connect(3002, "http")
print(f"\n[PAN-AI] Tunnel sukses terbentuk! URL publik Anda adalah: {public_url.public_url}\n")
print("Gunakan URL ini di berkas .env Express Backend sebagai PAN_AI_URL.")

## 5. Menjalankan Server FastAPI

Jalankan sel ini untuk memulai server FastAPI di Google Colab. Server akan mendengarkan request dari backend Express lokal Anda via ngrok.

In [ ]:
import uvicorn
from server_app import app

print("[PAN-AI] Memulai server di port 3002...")
uvicorn.run(app, host="0.0.0.0", port=3002)